
# Сбор и обработка данных о вакансиях через API trudvsem.ru

## Описание проекта

Проект посвящён сбору данных о вакансиях с портала «Работа России» (trudvsem.ru) с использованием открытого API.

В рамках проекта реализован процесс получения данных о вакансиях Ленинградской области, извлечения необходимых характеристик, обработки и сохранения результатов в CSV-файл.

## Задачи проекта

- Изучить структуру и возможности API trudvsem.ru.
- Реализовать отправку GET-запросов и получение данных в формате JSON.
- Организовать постраничную загрузку данных с использованием параметров пагинации.
- Извлечь основные характеристики вакансий: название должности, регион, работодателя, зарплату, тип занятости и график работы.
- Сохранить собранные данные в CSV-файл.
- Проверить данные на пропуски и дубликаты.

## Используемые инструменты

- Python
- Requests
- Pandas
- JSON
- CSV

## Результат

В результате реализован скрипт для автоматизированного сбора данных через API. Полученный набор данных содержит 9 780 уникальных вакансий после удаления дубликатов.


In [1]:
import requests
import pandas as pd
import numpy as np
import csv
import os

from tqdm import tqdm

In [2]:
# 1 функция для get запроса
def send_get_request(api_url: str, params_api: dict = None) -> dict:
    """
    Функция отправляет get запрос по указанному адресу с заданными параметрами

    Args: url: str -- адрес API
    params_api: dict -- параметры запроса

    Return:
    функция возвращает ответ с сервера в формате json
    """
    return requests.get(api_url, params=params_api).json()

# 2 список вакансий

def get_list_vac(all_vac:dict)->list:
    """
    Функция извлекает список вакансий
    """
    return all_vac.get('results',{}).get('vacancies', [])

# 3 разбор ответа
def get_vacancy(vac_info: dict) -> dict:
    """ Выбирает нужные поля каждой вакансии   """
    dct_vac = dict.fromkeys(['vac_id', 'vac_region_name','vac_company_name', 'vac_company_inn',
                             'vac_company_salary','vac_salary_min', 'vac_salary_max',
                             'vac_job_name', 'vac_employment', 'vac_schedule',
                             'vac_contact_person', 'vac_contact_phone'], None)
    dct_vac['vac_id'] = vac_info.get('vacancy').get('id')
    dct_vac['vac_region_name']  = vac_info.get('vacancy').get('region').get('name', 'регион не указан')
    dct_vac['vac_company_name'] = vac_info.get('vacancy').get('company').get('name', 'нет данных')
    dct_vac['vac_company_inn'] = vac_info.get('vacancy').get('company').get('inn', 'нет данных')
    dct_vac['vac_company_salary'] = vac_info.get('vacancy').get('salary', 'нет данных')
    dct_vac['vac_salary_min'] = vac_info.get('vacancy').get('salary_min', 'нет данных')
    dct_vac['vac_salary_max'] = vac_info.get('vacancy').get('salary_max', 'нет данных')
    dct_vac['vac_job_name'] = vac_info.get('vacancy').get('job-name', 'нет данных')
    dct_vac['vac_employment'] = vac_info.get('vacancy').get('employment', 'нет данных')
    dct_vac['vac_schedule'] = vac_info.get('vacancy').get('schedule', 'нет данных')
    dct_vac['vac_contact_person'] = vac_info.get('vacancy').get('contact_person', 'нет данных')
    dct_vac['vac_contact_phone'] = (
    vac_info.get('vacancy', {}).get('contact_list', [{}])[0].get('contact_value', 'нет данных')
    if vac_info.get('vacancy', {}).get('contact_list') else 'нет данных'
)

    return dct_vac.copy()

# 4 функция для записи в файл
name_file = 'vacancies_hw.csv'
def write_book_to_file(name_file:str, dct_vac:dict):
    # сделать проверку, если файл существует, то записываем переданную строку
    # если файл не существует, то создаем и записываем строку с заголовками
    if os.path.isfile(name_file):
        with open(name_file, 'a') as csvfile:
            fieldnames = list(dct_vac.keys())
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writerow(dct_vac)
    else:
        with open(name_file, 'a') as csvfile:
            fieldnames = list(dct_vac.keys())
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerow(dct_vac)

# 5 вычисление количества страниц (offset шагов)
def get_offset_count(api_url: str, params_api: dict) -> int:
    response = send_get_request(api_url, params_api)
    total = response.get('meta', {}).get('total', 0)
    limit = response.get('meta', {}).get('limit', 100)
    return total // limit + 1 if limit else 0


# 6 функция запуск процесс
def pipeline_vac(name_file, offset_full: int, api_url:str, params_api:dict=None):
    # Считаем количество страниц
    if offset_full is None:
        offset_full = get_offset_count(api_url, params_api)
    limit = params_api.get('limit', 100)

    print(f'Всего нужно пройти {offset_full} страниц по {limit} записей.\n')


# for offset_value in tqdm(range(0, offset_full)):
# Проход по всем страницам с шагом limit
    for page in tqdm(range(offset_full)):
        offset = page
        params_api.update({'offset': offset})
        # print(f'Обработка offset = {offset}...')

        response = send_get_request(api_url, params_api)
        list_all_vac = get_list_vac(response)

        for vac in list_all_vac:
            dct_vac = get_vacancy(vac)
            write_book_to_file(name_file, dct_vac)

    print('\n✅ Данные успешно собраны.')


In [15]:
# Проверка пагинации API

test_params = {
    'region_code': 47,
    'offset': 0,
    'limit': 100
}

response = send_get_request(
    'http://opendata.trudvsem.ru/api/v1/vacancies',
    test_params
)

print(response.get('meta'))

for offset in [0, 1, 2, 3, 5]:
    test_params['offset'] = offset

    response = send_get_request(
        'http://opendata.trudvsem.ru/api/v1/vacancies',
        test_params
    )

    vacancies = get_list_vac(response)

    print(f'Offset: {offset}, записей: {len(vacancies)}')

{'total': 9782, 'limit': 100}
Offset: 0, записей: 100
Offset: 1, записей: 100
Offset: 2, записей: 100
Offset: 3, записей: 100
Offset: 5, записей: 100


In [ ]:
params_api = {
    'region_code' : 47,
    'offset' : 0,
    'limit': 100}

pipeline_vac(
    name_file = 'vacancies_hw.csv',
    offset_full=None,
    api_url='http://opendata.trudvsem.ru/api/v1/vacancies',
    params_api=params_api
)


In [4]:
from tqdm import tqdm

for offset in tqdm(range(62, 98)):
    params_api['offset'] = offset

    response = send_get_request(
        'http://opendata.trudvsem.ru/api/v1/vacancies',
        params_api
    )

    vacancies = get_list_vac(response)

    for vac in vacancies:
        dct_vac = get_vacancy(vac)
        write_book_to_file('vacancies_hw.csv', dct_vac)

print('Сбор завершён!')

100%|██████████| 36/36 [10:16<00:00, 17.14s/it]

Сбор завершён!


In [5]:
# проверим наличие дубликатов в собранных данных
df = pd.read_csv('vacancies_hw.csv')

print('Всего строк:', len(df))
print('Уникальных вакансий:', df['vac_id'].nunique())
print('Дубликатов:', df['vac_id'].duplicated().sum())

Всего строк: 9782
Уникальных вакансий: 9780
Дубликатов: 2


In [6]:
# найдем эти дубли
duplicates = df[df['vac_id'].duplicated(keep=False)].sort_values('vac_id')
duplicates

,vac_id,vac_region_name,vac_company_name,vac_company_inn,vac_company_salary,vac_salary_min,vac_salary_max,vac_job_name,vac_employment,vac_schedule,vac_contact_person,vac_contact_phone
2999,2222bb28-7ec3-11f1-baa9-e58b42ebc806,Ленинградская область,"ГБУЗ ЛО ""ТОКСОВСКАЯ КМБ""",4703011525,от 68000,68000,68000,Рентгенолаборант,нет данных,Полный рабочий день,Маркова Евгения,+7(951) 654-15-50
3000,2222bb28-7ec3-11f1-baa9-e58b42ebc806,Ленинградская область,"ГБУЗ ЛО ""ТОКСОВСКАЯ КМБ""",4703011525,от 68000,68000,68000,Рентгенолаборант,нет данных,Полный рабочий день,Маркова Евгения,+7(951) 654-15-50
4499,4b025648-9af7-11f1-800d-3160d0db64b8,Ленинградская область,"ПАО ""ВЫМПЕЛКОМ""",7713076301,от 60000,60000,78000,Продавец-консультант,Полная занятость,Сменная работа,Мартыненко Снежана Александровна,+7(969) 719-17-90
4500,4b025648-9af7-11f1-800d-3160d0db64b8,Ленинградская область,"ПАО ""ВЫМПЕЛКОМ""",7713076301,от 60000,60000,78000,Продавец-консультант,Полная занятость,Сменная работа,Мартыненко Снежана Александровна,+7(969) 719-17-90


In [7]:
# удалим дубли
df = df.drop_duplicates(subset='vac_id').reset_index(drop=True)

In [8]:
# Проверим результат:
print('Всего строк:', len(df))
print('Уникальных вакансий:', df['vac_id'].nunique())
print('Дубликатов:', df['vac_id'].duplicated().sum())

Всего строк: 9780
Уникальных вакансий: 9780
Дубликатов: 0


In [9]:
# И сохраним очищенный файл
df.to_csv('vacancies_hw.csv', index=False)

In [10]:
# проверим структуру итогового DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9780 entries, 0 to 9779
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   vac_id              9780 non-null   object
 1   vac_region_name     9778 non-null   object
 2   vac_company_name    9780 non-null   object
 3   vac_company_inn     9780 non-null   int64 
 4   vac_company_salary  9780 non-null   object
 5   vac_salary_min      9780 non-null   int64 
 6   vac_salary_max      9780 non-null   int64 
 7   vac_job_name        9780 non-null   object
 8   vac_employment      9780 non-null   object
 9   vac_schedule        9780 non-null   object
 10  vac_contact_person  9780 non-null   object
 11  vac_contact_phone   9780 non-null   object
dtypes: int64(3), object(9)
memory usage: 917.0+ KB


In [11]:
df.head()

,vac_id,vac_region_name,vac_company_name,vac_company_inn,vac_company_salary,vac_salary_min,vac_salary_max,vac_job_name,vac_employment,vac_schedule,vac_contact_person,vac_contact_phone
0,9fd1e528-802e-11f1-9d9f-7de008f3dac7,Ленинградская область,"ГБУЗ ЛО ""СЕРТОЛОВСКАЯ ГБ""",4703036520,от 70000,70000,70000,Медицинская сестра по массажу,нет данных,Полный рабочий день,Нягина Любовь Васильевна,+7(812) 655-12-11
1,70063b28-8033-11f1-85d0-d789855dc8d1,Ленинградская область,"ГБУЗ ЛО ""СЕРТОЛОВСКАЯ ГБ""",4703036520,от 50000,50000,60000,Медицинская сестра процедурной,Полная занятость,Полный рабочий день,Нягина Любовь Васильевна,+7(812) 646-02-63
2,cf21cf74-877c-11f1-accb-878203fa1d89,Ленинградская область,Пятёрочка,7702319337,от 59900,59900,76600,"Продавец-кассир (Куйвози, Куйвози, Александров...",нет данных,Режим гибкого рабочего времени,Подбор персонала Пятёрочка,+7 495 1832415
3,6967ca60-a292-11f0-a18c-bdc82d0d5dcd,Ленинградская область,"АО ""ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ЛЕНИНГРАДСКАЯ ОБ...",4700000109,от 50000,50000,60000,Специалист персонального обслуживания,Полная занятость,Полный рабочий день,Павлова Лариса Васильевна,+7(931) 005-64-49
4,e6d2aec8-e328-11f1-9c45-e5a4da9c0dad,Ленинградская область,"АО ""СЗППК""",7839330845,от 90000,90000,90000,контролер-кассир билетный (разъездной) станция...,Полная занятость,Сменная работа,Пахомова Татьяна Николаевна,+7(931) 370-06-89


Offset: 0, записей: 100
Offset: 1, записей: 100
Offset: 2, записей: 100
Offset: 3, записей: 100
Offset: 5, записей: 100


{'total': 9746, 'limit': 100}


Всего нужно пройти 116 страниц по 100 записей.



100%|██████████| 116/116 [00:48<00:00,  2.39it/s]


✅ Данные успешно собраны.
